# Time

This notebook will contain the information regarding:

- how to embedd time as a featue
- Which aggregation levels to use
- window sizes to use



In [1]:
# Main packages 
import polars as pl
import matplotlib.pyplot as plt
import numpy as np

# Clustering packages
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

In [2]:
# Load data
df = pl.scan_csv("/home/lanl/data/cyber1/auth.txt.gz",has_header=False,separator=",",
                 new_columns= ['time','src_user','dest_user','src_comp','dest_comp',
                               'auth_type','logon_type','auth_orientation','outcome'])

In [3]:
# Human users
df = df.filter(pl.col("src_user").str.starts_with("U"))

In [4]:
# First 5 rows of the data
df.show(5)

time,src_user,dest_user,src_comp,dest_comp,auth_type,logon_type,auth_orientation,outcome
i64,str,str,str,str,str,str,str,str
1,"""U101@DOM1""","""C1862$@DOM1""","""C1862""","""C1862""","""?""","""?""","""AuthMap""","""Success"""
1,"""U101@DOM1""","""U101@DOM1""","""C1862""","""C1862""","""Negotiate""","""Interactive""","""LogOn""","""Success"""
1,"""U10@DOM1""","""U10@DOM1""","""C229""","""C229""","""Kerberos""","""Network""","""LogOn""","""Success"""
1,"""U10@DOM1""","""U10@DOM1""","""C62""","""C528""","""Kerberos""","""Network""","""LogOn""","""Success"""
1,"""U1137@DOM1""","""U1137@DOM1""","""C1065""","""C1065""","""?""","""Network""","""LogOff""","""Success"""


## Embedding time

number of seconds in a day = $60 * 60 * 24 = 86400$

$ \theta_i = \frac{t_i mod 86400}{86400} * 2 \pi$.

$ \theta_i$ is the angle on the clock.

$(\cos(\theta_i), \sin(\theta_i))$ maps each timestamp to a point on the unit cirle.

$\bar{C} = \frac{1}{n} \sum_{i=1}^{n} \cos{\theta_i}$. $\bar{S} = \frac{1}{n} \sum_{i=1}^{n} \sin{\theta_i}$.


The features will then be $(\bar{C},\bar{S})$



Warning: when using standard scalar, we distort the geometry of the circle slightly.


Reference: Mardia & Jupp, Directional Statistics (2000) Page 15


Statistical Analysis of Circular Data -- Nicholas I_ Fisher -- 1, 1993 -- Cambridge University Press (Virtual Publishing).
Page 31

Why: To ensure that 23:00 and 01:00 am are 2 hours apart and not 22 hours if using distance.



In [5]:
# Features dataframe
AGG_HOURS = 1
SECONDS_IN_DAY = 86400
AGG_SECONDS = AGG_HOURS * 3600

features = (
    df
    .with_columns(
        bucket=pl.col("time") // AGG_SECONDS,
        theta=(pl.col("time") % SECONDS_IN_DAY) / SECONDS_IN_DAY * 2 * np.pi,
        is_failure=(pl.col("outcome") != "Success").cast(pl.Int8),
    )
    .group_by(["src_user", "bucket"])
    .agg(
        n_events=pl.len(),
        failure_ratio=pl.col("is_failure").mean(),
        n_distinct_dest=pl.col("dest_comp").n_unique(),
        n_distinct_src=pl.col("src_comp").n_unique(),
        C_bar=pl.col("theta").cos().mean(),
        S_bar=pl.col("theta").sin().mean(),
    ).with_columns(
    log_n_events=pl.col("n_events").log1p(),
    log_n_distinct_dest=pl.col("n_distinct_dest").log1p(),
    log_n_distinct_src=pl.col("n_distinct_src").log1p(),
)
    .collect()
)

In [6]:
features.show(5)

src_user,bucket,n_events,failure_ratio,n_distinct_dest,n_distinct_src,C_bar,S_bar,log_n_events,log_n_distinct_dest,log_n_distinct_src
str,i64,u64,f64,u64,u64,f64,f64,f64,f64,f64
"""U7333@DOM1""",627,28,0.0,6,4,0.689543,0.722712,3.367296,1.94591,1.609438
"""U5571@DOM1""",1383,14,0.0,2,3,-0.568312,-0.818076,2.70805,1.098612,1.386294
"""U10023@DOM90""",154,3,0.0,1,1,-0.958385,0.284264,1.386294,0.693147,0.693147
"""U5654@C13252""",436,1,1.0,1,1,0.425977,0.904734,0.693147,0.693147,0.693147
"""U5049@DOM1""",213,16,0.0,4,5,0.812933,-0.582355,2.833213,1.609438,1.791759
